# Automatic construction of clinical NLP pre-annotations

This notebook reproduces the deterministic part of the annotation workflow.

It performs the following operations:

1. loads the fixed cohort of 50 encounters from the cohort manifest;
2. reads anamnesis, admission therapy, and discharge therapy reports;
3. extracts regularly formatted prescriptions;
4. calculates character offsets;
5. generates stable annotation identifiers;
6. computes SHA-256 hashes for source reports;
7. derives a product-to-active-ingredient lexicon from explicit discharge headers;
8. writes tabular and JSONL pre-annotations;
9. verifies that every annotated span matches the original source text.

The generated files are **pre-annotations**. Ambiguous mappings, malformed entries, and non-standard instructions are placed in review queues and must be checked before they are promoted to the final gold standard.

## Project layout

The notebook is stored in `notebooks/`.

Input files:

- `data/pazienti_con_terapia_uscita_testuale.json`
- `data/annotations/gold_report_manifest.csv`

Generated files are written to:

- `data/annotations/generated/`

In [1]:
from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

In [2]:
def find_project_root(start: Path) -> Path:
    """Find the repository root containing data/annotations."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "annotations").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find data/annotations. Run the notebook inside the project repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
ANNOTATION_DIR = PROJECT_ROOT / "data" / "annotations"
OUTPUT_DIR = ANNOTATION_DIR / "generated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = PROJECT_ROOT / "data" / "pazienti_con_terapia_uscita_testuale.json"
COHORT_MANIFEST_PATH = ANNOTATION_DIR / "gold_report_manifest.csv"

for required_path in [DATASET_PATH, COHORT_MANIFEST_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required file: {required_path}")

print("Project directories initialized.")
print(f"Input directory: {ANNOTATION_DIR.relative_to(PROJECT_ROOT)}")
print(f"Output directory: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")

Project directories initialized.
Input directory: data\annotations
Output directory: data\annotations\generated


## Constants and helper functions

Offsets use Python slicing conventions:

- `start` is inclusive;
- `end` is exclusive;
- `source_text[start:end]` must equal the annotated text.

In [3]:
REPORT_ANAMNESIS = "Anamnesi"
REPORT_INGRESS = "Terapia medica all'ingresso"
REPORT_DISCHARGE = "Terapia alla Dimissione"
REQUIRED_REPORT_TYPES = [
    REPORT_ANAMNESIS,
    REPORT_INGRESS,
    REPORT_DISCHARGE,
]

QUOTED_ENTRY_RE = re.compile(r'"(?P<entry>[^"]+)"')
DISCHARGE_HEADER_RE = re.compile(
    r"^\s*(?P<ingredient>[^(:]+?)\s*"
    r"(?:\((?P<product>[^)]*)\))?\s*:\s*"
    r"(?P<instruction>.*)$",
    flags=re.DOTALL,
)

FORM_TOKEN_RE = re.compile(
    r"\b("
    r"cp|cpr|cps|compressa|compresse|capsula|capsule|"
    r"cerotto|cerotti|soluz|soluzione|sosp|sospensione|"
    r"gtt|gocce|fiala|fiale|flacone|spray|collirio|"
    r"crema|pomata|granulato|polvere|polv|bustina|bustine|"
    r"nebuliz|inalaz|iniett"
    r")\b",
    flags=re.IGNORECASE,
)

NON_DRUG_RE = re.compile(
    r"\b("
    r"ossigeno|niv|cpap|ventilazione|maschera|cannula|"
    r"simeox|supporto respiratorio"
    r")\b",
    flags=re.IGNORECASE,
)

NO_THERAPY_RE = re.compile(
    r"\bnessuna\s+terapia(?:\s+domiciliare)?\b",
    flags=re.IGNORECASE,
)


def normalize_key(text: str) -> str:
    """Normalize a surface form for dictionary lookup."""
    text = str(text).lower().strip()
    text = "".join(
        character
        for character in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(character)
    )
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def sha256_text(text: str) -> str:
    """Return the SHA-256 digest of a UTF-8 text."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def get_report(patient: dict[str, Any], report_type: str) -> dict[str, Any]:
    """Return one report of the requested type."""
    matches = [
        report
        for report in patient.get("referti", [])
        if report.get("tipo") == report_type
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Encounter {patient.get('encOid')} has {len(matches)} reports of type {report_type!r}."
        )
    return matches[0]


def trimmed_group_span(match: re.Match[str], group_name: str) -> tuple[int, int, str]:
    """Return the trimmed local span and value of a regular-expression group."""
    raw_value = match.group(group_name)
    if raw_value is None:
        return -1, -1, ""

    left_trim = len(raw_value) - len(raw_value.lstrip())
    right_trim = len(raw_value) - len(raw_value.rstrip())
    start = match.start(group_name) + left_trim
    end = match.end(group_name) - right_trim
    return start, end, raw_value.strip()


def split_active_ingredients(label: str) -> list[str]:
    """Split combination products written with slash-separated ingredients."""
    return [
        item.strip()
        for item in str(label).split("/")
        if item.strip()
    ]


def extract_product_brand(product_text: str) -> str:
    """Remove formulation and strength information from a product description."""
    product_text = str(product_text).strip()
    if not product_text:
        return ""

    form_match = FORM_TOKEN_RE.search(product_text)
    if form_match:
        return product_text[:form_match.start()].strip()

    return re.sub(r"\s+\d[\d.,/]*.*$", "", product_text).strip()


def prescription_status(instruction: str) -> str:
    """Infer a simple prescription status from explicit textual cues."""
    lowered = instruction.casefold()

    if re.search(r"\bsospes[oaie]\b", lowered):
        return "suspended"
    if "al bisogno" in lowered:
        return "active_prn"
    if re.search(r"\bpoi\s+stop\b", lowered) or (
        "per altri" in lowered and "stop" in lowered
    ):
        return "active_planned_stop"
    return "active"


def dataframe_records(frame: pd.DataFrame) -> list[dict[str, Any]]:
    """Convert a DataFrame to JSON-safe records."""
    return json.loads(frame.to_json(orient="records", force_ascii=False))

## Load and validate the cohort

The cohort manifest fixes the encounter identifiers and their order. The dataset is not resampled inside this notebook.

In [4]:
patients = json.loads(DATASET_PATH.read_text(encoding="utf-8"))

if not isinstance(patients, list) or not patients:
    raise ValueError("The dataset must be a non-empty JSON list.")

patients_by_id = {
    int(patient["encOid"]): patient
    for patient in patients
}

if len(patients_by_id) != len(patients):
    raise ValueError("Duplicate encOid values were found in the dataset.")

cohort_manifest = pd.read_csv(
    COHORT_MANIFEST_PATH,
    dtype={
        "encOid": "int64",
        "reportOid": "int64",
        "report_type": "string",
        "sha256": "string",
    },
)

selected_ids = (
    cohort_manifest["encOid"]
    .drop_duplicates()
    .tolist()
)

if len(selected_ids) != len(set(selected_ids)):
    raise ValueError(
        "The cohort manifest contains duplicate encounter IDs."
    )

missing_ids = sorted(
    set(selected_ids) - set(patients_by_id)
)

if missing_ids:
    raise KeyError(
        f"Encounter IDs missing from the dataset: {missing_ids}"
    )

selected_patients = [
    patients_by_id[encounter_id]
    for encounter_id in selected_ids
]

print(f"Dataset encounters: {len(patients):,}")
print(f"Selected encounters: {len(selected_patients)}")
print(f"Manifest rows: {len(cohort_manifest)}")

Dataset encounters: 857
Selected encounters: 50
Manifest rows: 150


## Build the source report manifest

The report manifest records the exact source text version used for annotation. SHA-256 is used as an integrity check because character offsets are only valid for the original text.

In [5]:
report_manifest_rows: list[dict[str, Any]] = []

for encounter_id in selected_ids:
    patient = patients_by_id[encounter_id]

    for report_type in REQUIRED_REPORT_TYPES:
        report = get_report(patient, report_type)
        text = report.get("testo") or ""

        report_manifest_rows.append(
            {
                "encOid": encounter_id,
                "report_type": report_type,
                "reportOid": int(report["reportOid"]),
                "report_date": report.get("data", ""),
                "text_length": len(text),
                "sha256": sha256_text(text),
            }
        )

report_manifest = (
    pd.DataFrame(report_manifest_rows)
    .sort_values(["encOid", "report_type"])
    .reset_index(drop=True)
)

expected_report_count = len(selected_ids) * len(REQUIRED_REPORT_TYPES)
assert len(report_manifest) == expected_report_count
assert not report_manifest.duplicated(["encOid", "report_type"]).any()

report_manifest.head()

,encOid,report_type,reportOid,report_date,text_length,sha256
0,9920559,Anamnesi,25385849,08.01.2026 08:20,1762,3f99b1d5bc2dcf587c213428f83694bd3a7d2539dd79dda924e6853be7781a8c
1,9920559,Terapia alla Dimissione,25412536,11.01.2026 11:45,302,7f70bb7d3194977c0a163d4d0f687ea42aadf571522d85d6d8e8ab8957c9c482
2,9920559,Terapia medica all'ingresso,25385973,08.01.2026 08:31,126,ccc6a3da1118842fbd5580252fd2f0624914cec21b23a17c538b71ba36076846
3,9992853,Anamnesi,25112711,27.11.2025 12:10,1058,04150ff2fc8c0a8d17ab5e220d254663a26cf13523f8e1da7386044c138da278
4,9992853,Terapia alla Dimissione,25402812,09.01.2026 14:35,1138,1ad71aef327b07e9802e9190c43628b976a47f8bc9c40518b4f89bf0caad52df


In [6]:
# Verify that the input manifest matches the reports reconstructed from the raw dataset.

MANIFEST_COLUMNS = [
    "encOid",
    "report_type",
    "reportOid",
    "report_date",
    "text_length",
    "sha256",
]


def normalize_manifest(frame: pd.DataFrame) -> pd.DataFrame:
    """Normalize manifest columns before an exact comparison."""
    normalized = frame[MANIFEST_COLUMNS].copy()

    normalized["encOid"] = normalized["encOid"].astype("int64")
    normalized["reportOid"] = normalized["reportOid"].astype("int64")
    normalized["text_length"] = normalized["text_length"].astype("int64")

    normalized["report_type"] = (
        normalized["report_type"]
        .fillna("")
        .astype("string")
    )
    normalized["report_date"] = (
        normalized["report_date"]
        .fillna("")
        .astype("string")
    )
    normalized["sha256"] = (
        normalized["sha256"]
        .fillna("")
        .astype("string")
    )

    return (
        normalized
        .sort_values(["encOid", "report_type"])
        .reset_index(drop=True)
    )


expected_manifest = normalize_manifest(cohort_manifest)
generated_manifest = normalize_manifest(report_manifest)

if expected_manifest.duplicated(
    ["encOid", "report_type"]
).any():
    raise AssertionError(
        "The input manifest contains duplicate encounter/report-type pairs."
    )

if generated_manifest.duplicated(
    ["encOid", "report_type"]
).any():
    raise AssertionError(
        "The reconstructed manifest contains duplicate encounter/report-type pairs."
    )

if len(expected_manifest) != len(generated_manifest):
    raise AssertionError(
        "Manifest row-count mismatch: "
        f"input={len(expected_manifest)}, "
        f"reconstructed={len(generated_manifest)}."
    )

comparison = expected_manifest.merge(
    generated_manifest,
    on=["encOid", "report_type"],
    how="outer",
    suffixes=("_expected", "_generated"),
    indicator=True,
)

value_columns = [
    "reportOid",
    "report_date",
    "text_length",
    "sha256",
]

mismatch_mask = comparison["_merge"].ne("both")

for column in value_columns:
    mismatch_mask |= (
        comparison[f"{column}_expected"].astype("string")
        != comparison[f"{column}_generated"].astype("string")
    )

mismatches = comparison.loc[mismatch_mask].copy()

if not mismatches.empty:
    display(mismatches)
    raise AssertionError(
        "The source reports do not match the input manifest. "
        "Check report identifiers, text lengths, or SHA-256 hashes."
    )

pd.testing.assert_frame_equal(
    expected_manifest,
    generated_manifest,
    check_dtype=False,
)

assert expected_manifest["encOid"].nunique() == 50
assert len(expected_manifest) == 150

print("Manifest verification passed.")
print("50 encounters and 150 reports match the raw dataset.")
print("All report identifiers, lengths, dates, and SHA-256 hashes are unchanged.")

Manifest verification passed.
50 encounters and 150 reports match the raw dataset.
All report identifiers, lengths, dates, and SHA-256 hashes are unchanged.


## Derive a product-to-ingredient lexicon

Discharge reports often contain an explicit structure such as:

```text
Active ingredient (commercial product and formulation): instruction
```

These headers provide supervision for normalizing the commercial forms found in admission therapy. The lexicon is derived from all available discharge reports to maximize coverage. Ambiguous product names are retained with their observed alternatives.

In [7]:
brand_votes: dict[str, Counter[str]] = defaultdict(Counter)
brand_examples: dict[tuple[str, str], str] = {}

for patient in patients:
    discharge_report = get_report(patient, REPORT_DISCHARGE)
    discharge_text = discharge_report.get("testo") or ""

    for quoted_match in QUOTED_ENTRY_RE.finditer(discharge_text):
        entry = quoted_match.group("entry")
        header_match = DISCHARGE_HEADER_RE.match(entry)

        if header_match is None or header_match.group("product") is None:
            continue

        _, _, ingredient_label = trimmed_group_span(
            header_match, "ingredient"
        )
        _, _, product_text = trimmed_group_span(
            header_match, "product"
        )
        brand = extract_product_brand(product_text)

        if not ingredient_label or not brand:
            continue

        normalized_brand = normalize_key(brand)
        if not normalized_brand:                 # e.g. a product text like "-" normalizes to empty
            continue
        brand_votes[normalized_brand][ingredient_label] += 1
        brand_examples[(normalized_brand, ingredient_label)] = brand

lexicon_rows: list[dict[str, Any]] = []

for normalized_brand, votes in brand_votes.items():
    total_count = sum(votes.values())
    ranked = votes.most_common()
    n_distinct_labels = len(votes)          # auto-resolvable only if the brand maps to ONE ingredient

    for rank, (ingredient_label, count) in enumerate(ranked, start=1):
        lexicon_rows.append(
            {
                "normalized_brand": normalized_brand,
                "brand_example": brand_examples[
                    (normalized_brand, ingredient_label)
                ],
                "ingredient_label": ingredient_label,
                "active_ingredients": json.dumps(
                    split_active_ingredients(ingredient_label),
                    ensure_ascii=False,
                ),
                "observation_count": count,
                "total_brand_observations": total_count,
                "relative_frequency": count / total_count,
                "rank": rank,
                "is_unique_top_candidate": n_distinct_labels == 1,
            }
        )

brand_lexicon = (
    pd.DataFrame(lexicon_rows)
    .sort_values(
        ["normalized_brand", "rank", "ingredient_label"]
    )
    .reset_index(drop=True)
)

# A brand is auto-resolved only when it was ever seen with a SINGLE distinct ingredient label; any
# brand observed with two or more ingredient labels is ambiguous and routed to manual review.
resolved_brand_map = {
    brand: next(iter(votes))
    for brand, votes in brand_votes.items()
    if len(votes) == 1
}

ambiguous_brand_keys = {
    brand for brand, votes in brand_votes.items() if len(votes) > 1
}

print(f"Observed normalized brands: {brand_lexicon['normalized_brand'].nunique():,}")
print(f"Unambiguous brand mappings: {len(resolved_brand_map):,}")
print(f"Ambiguous brand keys: {len(ambiguous_brand_keys):,}")

brand_lexicon.head(10)

Observed normalized brands: 777
Unambiguous brand mappings: 771
Ambiguous brand keys: 6


,normalized_brand,brand_example,ingredient_label,active_ingredients,observation_count,total_brand_observations,relative_frequency,rank,is_unique_top_candidate
0,2 5 mg,2.5 mg,Enalapril,"[""Enalapril""]",1,1,1.0,1,True
1,abesart,Abesart,Irbesartan,"[""Irbesartan""]",2,2,1.0,1,True
2,abigerd,Abigerd,Esomeprazolo,"[""Esomeprazolo""]",7,7,1.0,1,True
3,abis,Abis,Amlodipina,"[""Amlodipina""]",2,2,1.0,1,True
4,absorcol,Absorcol,Ezetimibe,"[""Ezetimibe""]",1,1,1.0,1,True
5,acarden,Acarden,Carvedilolo,"[""Carvedilolo""]",1,1,1.0,1,True
6,aciclin,Aciclin,Aciclovir,"[""Aciclovir""]",1,1,1.0,1,True
7,acido acetils au,Acido acetils au,Acido acetilsalicilico,"[""Acido acetilsalicilico""]",13,13,1.0,1,True
8,acido acetils eg,Acido acetils eg,Acido acetilsalicilico,"[""Acido acetilsalicilico""]",81,81,1.0,1,True
9,acido acetils san,Acido acetils san,Acido acetilsalicilico,"[""Acido acetilsalicilico""]",4,4,1.0,1,True


In [8]:
# Direct generic names are also collected from explicit discharge labels.
direct_ingredient_map: dict[str, str] = {}

for ingredient_label in brand_lexicon["ingredient_label"].drop_duplicates():
    for ingredient in split_active_ingredients(ingredient_label):
        direct_ingredient_map.setdefault(
            normalize_key(ingredient),
            ingredient,
        )


def resolve_ingredient_surface(
    surface: str,
) -> tuple[list[str], str, str]:
    """Resolve an admission-therapy surface form using data-derived evidence."""
    key = normalize_key(surface)

    if key in resolved_brand_map:
        canonical_label = resolved_brand_map[key]
        return (
            split_active_ingredients(canonical_label),
            "resolved_from_discharge_header",
            canonical_label,
        )

    if key in ambiguous_brand_keys:
        return [], "ambiguous_product_name", ""

    if key in direct_ingredient_map:
        ingredient = direct_ingredient_map[key]
        return [ingredient], "direct_active_ingredient", ingredient

    return [], "unresolved", ""

## Extract discharge prescriptions

Only regularly formatted quoted prescriptions are extracted automatically. Unparsed entries and non-drug therapies are saved separately for manual review.

In [9]:
discharge_rows: list[dict[str, Any]] = []
discharge_review_rows: list[dict[str, Any]] = []

for encounter_id in selected_ids:
    report = get_report(
        patients_by_id[encounter_id],
        REPORT_DISCHARGE,
    )
    source_text = report.get("testo") or ""
    quoted_matches = list(QUOTED_ENTRY_RE.finditer(source_text))

    if not quoted_matches:
        reason = (
            "explicit_no_therapy"
            if NO_THERAPY_RE.search(source_text)
            else "no_regular_quoted_entries"
        )
        discharge_review_rows.append(
            {
                "encOid": encounter_id,
                "reportOid": int(report["reportOid"]),
                "reason": reason,
                "source_text": source_text,
            }
        )
        continue

    accepted_entries: list[dict[str, Any]] = []

    for quoted_match in quoted_matches:
        entry = quoted_match.group("entry")
        entry_start = quoted_match.start("entry")
        entry_end = quoted_match.end("entry")
        header_match = DISCHARGE_HEADER_RE.match(entry)

        if header_match is None:
            discharge_review_rows.append(
                {
                    "encOid": encounter_id,
                    "reportOid": int(report["reportOid"]),
                    "reason": "unparsed_quoted_entry",
                    "source_text": entry,
                }
            )
            continue

        local_start, local_end, ingredient_surface = trimmed_group_span(
            header_match,
            "ingredient",
        )
        _, _, product_text = trimmed_group_span(
            header_match,
            "product",
        )
        _, _, instruction_text = trimmed_group_span(
            header_match,
            "instruction",
        )

        if not ingredient_surface:
            discharge_review_rows.append(
                {
                    "encOid": encounter_id,
                    "reportOid": int(report["reportOid"]),
                    "reason": "missing_ingredient_header",
                    "source_text": entry,
                }
            )
            continue

        if NON_DRUG_RE.search(ingredient_surface):
            discharge_review_rows.append(
                {
                    "encOid": encounter_id,
                    "reportOid": int(report["reportOid"]),
                    "reason": "non_drug_therapy",
                    "source_text": entry,
                }
            )
            continue

        mention_start = entry_start + local_start
        mention_end = entry_start + local_end
        status = prescription_status(instruction_text)

        accepted_entries.append(
            {
                "encOid": encounter_id,
                "reportOid": int(report["reportOid"]),
                "report_date": report.get("data", ""),
                "entry_start": entry_start,
                "entry_end": entry_end,
                "mention_start": mention_start,
                "mention_end": mention_end,
                "mention_text": source_text[
                    mention_start:mention_end
                ],
                "active_ingredients": json.dumps(
                    split_active_ingredients(ingredient_surface),
                    ensure_ascii=False,
                ),
                "product_text": product_text,
                "instruction_text": instruction_text,
                "prescription_status": status,
                "include_in_current_regimen": status != "suspended",
                "annotation_source": "regular_discharge_header",
            }
        )

    accepted_entries.sort(key=lambda row: row["mention_start"])

    for local_index, row in enumerate(accepted_entries):
        row["annotation_id"] = (
            f"DIS-{encounter_id}-{local_index:03d}"
        )
        discharge_rows.append(row)

discharge_preannotations = (
    pd.DataFrame(discharge_rows)
    .sort_values(["encOid", "mention_start"])
    .reset_index(drop=True)
)

discharge_review_queue = pd.DataFrame(discharge_review_rows)

print(f"Automatic discharge annotations: {len(discharge_preannotations):,}")
print(f"Discharge review items: {len(discharge_review_queue):,}")

discharge_preannotations.head()

Automatic discharge annotations: 371
Discharge review items: 6


,encOid,reportOid,report_date,entry_start,entry_end,mention_start,mention_end,mention_text,active_ingredients,product_text,instruction_text,prescription_status,include_in_current_regimen,annotation_source,annotation_id
0,9920559,25412536,11.01.2026 11:45,1,64,1,12,Rivaroxaban,"[""Rivaroxaban""]",Mirebax cp.riv. 20 mg,da assumere 20 mg (ore 20),active,True,regular_discharge_header,DIS-9920559-000
1,9920559,25412536,11.01.2026 11:45,67,144,67,91,Dolutegravir/rilpivirina,"[""Dolutegravir"", ""rilpivirina""]",Juluca cp.riv. 25 mg,da assumere 25/50 mg (ore 8),active,True,regular_discharge_header,DIS-9920559-001
2,9992853,25402812,09.01.2026 14:35,1,74,1,13,Levotiroxina,"[""Levotiroxina""]",Eutirox cpr. divisibili 50 mcg,da assumere 50 mcg (ore 8),active,True,regular_discharge_header,DIS-9992853-000
3,9992853,25402812,09.01.2026 14:35,77,138,77,88,Linagliptin,"[""Linagliptin""]",Trajenta cp.riv. 5 mg,da assumere 5 mg (ore 8),active,True,regular_discharge_header,DIS-9992853-001
4,9992853,25402812,09.01.2026 14:35,141,215,141,153,Allopurinolo,"[""Allopurinolo""]",Zyloric cpr. divisibili 300 mg,da assumere 150 mg (ore 12),active,True,regular_discharge_header,DIS-9992853-002


## Extract admission-therapy mentions

Regular admission therapy is usually semicolon-separated, with the surface form placed before the first colon. Product names are normalized through the lexicon derived above.

Unresolved or ambiguous mentions are kept in the output and copied to a review queue.

In [10]:
def iter_semicolon_segments(text: str):
    """Yield trimmed semicolon-separated segments with source offsets."""
    for match in re.finditer(r"[^;]+(?:;|$)", text):
        raw_segment = match.group(0)
        without_separator = (
            raw_segment[:-1]
            if raw_segment.endswith(";")
            else raw_segment
        )

        left_trim = len(without_separator) - len(
            without_separator.lstrip()
        )
        right_boundary = len(without_separator.rstrip())

        start = match.start() + left_trim
        end = match.start() + right_boundary

        if start < end:
            yield start, end, text[start:end]


ingress_rows: list[dict[str, Any]] = []
ingress_review_rows: list[dict[str, Any]] = []

for encounter_id in selected_ids:
    report = get_report(
        patients_by_id[encounter_id],
        REPORT_INGRESS,
    )
    source_text = report.get("testo") or ""

    if NO_THERAPY_RE.search(source_text):
        ingress_review_rows.append(
            {
                "encOid": encounter_id,
                "reportOid": int(report["reportOid"]),
                "reason": "explicit_no_therapy",
                "source_text": source_text,
            }
        )
        continue

    accepted_mentions: list[dict[str, Any]] = []

    for segment_start, segment_end, segment in iter_semicolon_segments(
        source_text
    ):
        if NON_DRUG_RE.search(segment):
            ingress_review_rows.append(
                {
                    "encOid": encounter_id,
                    "reportOid": int(report["reportOid"]),
                    "reason": "non_drug_therapy",
                    "source_text": segment,
                }
            )
            continue

        colon_position = segment.find(":")
        if colon_position < 0:
            ingress_review_rows.append(
                {
                    "encOid": encounter_id,
                    "reportOid": int(report["reportOid"]),
                    "reason": "missing_colon_or_non_standard_list",
                    "source_text": segment,
                }
            )
            continue

        raw_surface = segment[:colon_position]
        left_trim = len(raw_surface) - len(raw_surface.lstrip())
        surface_text = raw_surface.strip()

        if not surface_text:
            ingress_review_rows.append(
                {
                    "encOid": encounter_id,
                    "reportOid": int(report["reportOid"]),
                    "reason": "empty_surface_form",
                    "source_text": segment,
                }
            )
            continue

        mention_start = segment_start + left_trim
        mention_end = mention_start + len(surface_text)
        instruction_text = segment[colon_position + 1:].strip()

        ingredients, mapping_status, canonical_label = (
            resolve_ingredient_surface(surface_text)
        )

        status = prescription_status(instruction_text)

        accepted_mentions.append(
            {
                "encOid": encounter_id,
                "reportOid": int(report["reportOid"]),
                "report_date": report.get("data", ""),
                "segment_start": segment_start,
                "segment_end": segment_end,
                "mention_start": mention_start,
                "mention_end": mention_end,
                "mention_text": source_text[
                    mention_start:mention_end
                ],
                "active_ingredients": json.dumps(
                    ingredients,
                    ensure_ascii=False,
                ),
                "canonical_label": canonical_label,
                "instruction_text": instruction_text,
                "medication_status": status,
                "mapping_status": mapping_status,
                "annotation_source": "regular_ingress_segment",
            }
        )

    accepted_mentions.sort(key=lambda row: row["mention_start"])

    for local_index, row in enumerate(accepted_mentions):
        row["annotation_id"] = (
            f"ING-{encounter_id}-{local_index:03d}"
        )
        ingress_rows.append(row)

        if row["mapping_status"] != "resolved_from_discharge_header" and (
            row["mapping_status"] != "direct_active_ingredient"
        ):
            ingress_review_rows.append(
                {
                    "encOid": encounter_id,
                    "reportOid": int(report["reportOid"]),
                    "reason": row["mapping_status"],
                    "source_text": row["mention_text"],
                }
            )

ingress_preannotations = (
    pd.DataFrame(ingress_rows)
    .sort_values(["encOid", "mention_start"])
    .reset_index(drop=True)
)

ingress_review_queue = pd.DataFrame(ingress_review_rows)

print(f"Automatic ingress annotations: {len(ingress_preannotations):,}")
print(f"Resolved ingress mappings: {(ingress_preannotations['mapping_status'].isin(['resolved_from_discharge_header', 'direct_active_ingredient'])).sum():,}")
print(f"Ingress review items: {len(ingress_review_queue):,}")

ingress_preannotations.head()

Automatic ingress annotations: 291
Resolved ingress mappings: 280
Ingress review items: 21


,encOid,reportOid,report_date,segment_start,segment_end,mention_start,mention_end,mention_text,active_ingredients,canonical_label,instruction_text,medication_status,mapping_status,annotation_source,annotation_id
0,9920559,25385973,08.01.2026 08:31,0,46,0,14,Propafenone eg,[],,300 mg cp.riv. /die al bisogno,active_prn,unresolved,regular_ingress_segment,ING-9920559-000
1,9920559,25385973,08.01.2026 08:31,48,85,48,54,Juluca,"[""Dolutegravir"", ""rilpivirina""]",Dolutegravir/rilpivirina,25/50 mg cp.riv. /die (ore 8),active,resolved_from_discharge_header,regular_ingress_segment,ING-9920559-001
2,9920559,25385973,08.01.2026 08:31,88,124,88,95,Mirebax,"[""Rivaroxaban""]",Rivaroxaban,20 mg cp.riv. /die (ore 20),active,resolved_from_discharge_header,regular_ingress_segment,ING-9920559-002
3,9992853,25112801,27.11.2025 12:18,0,106,0,7,Humalog,[],,5 U soluz. iniett. /die (ore 12) 5 U soluz. iniett. /die (ore 8) 5 U soluz. iniett. /die (ore 19),active,ambiguous_product_name,regular_ingress_segment,ING-9992853-000
4,9992853,25112801,27.11.2025 12:18,109,153,109,116,Eutirox,"[""Levotiroxina""]",Levotiroxina,50 mcg cpr. divisibili /die (ore 8),active,resolved_from_discharge_header,regular_ingress_segment,ING-9992853-001


## Validate identifiers and offsets

The main integrity rule is checked directly against the raw reports:

```python
source_text[start:end] == mention_text
```

In [11]:
def source_text_for(
    encounter_id: int,
    report_type: str,
) -> str:
    report = get_report(
        patients_by_id[int(encounter_id)],
        report_type,
    )
    return report.get("testo") or ""


def check_offsets(
    frame: pd.DataFrame,
    report_type: str,
) -> bool:
    for row in frame.itertuples(index=False):
        source_text = source_text_for(
            int(row.encOid),
            report_type,
        )
        extracted = source_text[
            int(row.mention_start):int(row.mention_end)
        ]
        if extracted != row.mention_text:
            raise AssertionError(
                f"Offset mismatch for {row.annotation_id}: "
                f"{extracted!r} != {row.mention_text!r}"
            )
    return True


assert discharge_preannotations["annotation_id"].is_unique
assert ingress_preannotations["annotation_id"].is_unique

discharge_offsets_valid = check_offsets(
    discharge_preannotations,
    REPORT_DISCHARGE,
)
ingress_offsets_valid = check_offsets(
    ingress_preannotations,
    REPORT_INGRESS,
)

assert discharge_offsets_valid
assert ingress_offsets_valid

print("All annotation identifiers are unique.")
print("All discharge offsets match the source text.")
print("All ingress offsets match the source text.")

All annotation identifiers are unique.
All discharge offsets match the source text.
All ingress offsets match the source text.


## Build the patient-level JSONL representation

CSV files are convenient for evaluation. JSONL keeps all generated annotations grouped by encounter and records the hashes of the three source reports.

In [12]:
discharge_by_encounter = {
    int(encounter_id): dataframe_records(group)
    for encounter_id, group in discharge_preannotations.groupby(
        "encOid",
        sort=False,
    )
}

ingress_by_encounter = {
    int(encounter_id): dataframe_records(group)
    for encounter_id, group in ingress_preannotations.groupby(
        "encOid",
        sort=False,
    )
}

jsonl_records: list[dict[str, Any]] = []

for encounter_id in selected_ids:
    patient = patients_by_id[encounter_id]

    reports = {}
    for report_type in REQUIRED_REPORT_TYPES:
        report = get_report(patient, report_type)
        text = report.get("testo") or ""
        reports[report_type] = {
            "reportOid": int(report["reportOid"]),
            "date": report.get("data", ""),
            "sha256": sha256_text(text),
        }

    jsonl_records.append(
        {
            "encOid": encounter_id,
            "reports": reports,
            "discharge_preannotations": discharge_by_encounter.get(
                encounter_id,
                [],
            ),
            "ingress_preannotations": ingress_by_encounter.get(
                encounter_id,
                [],
            ),
        }
    )

assert len(jsonl_records) == len(selected_ids)

## Save generated files

The output names use the `auto_` prefix to distinguish deterministic pre-annotations from manually reviewed gold files.

In [13]:
output_paths = {
    "report_manifest": OUTPUT_DIR / "auto_report_manifest.csv",
    "brand_lexicon": OUTPUT_DIR / "auto_brand_ingredient_lexicon.csv",
    "discharge_annotations": OUTPUT_DIR / "auto_discharge_medications.csv",
    "discharge_review": OUTPUT_DIR / "auto_discharge_review_queue.csv",
    "ingress_annotations": OUTPUT_DIR / "auto_ingress_medications.csv",
    "ingress_review": OUTPUT_DIR / "auto_ingress_review_queue.csv",
    "jsonl": OUTPUT_DIR / "auto_annotations.jsonl",
    "checks": OUTPUT_DIR / "auto_checks.json",
}

report_manifest.to_csv(
    output_paths["report_manifest"],
    index=False,
)
brand_lexicon.to_csv(
    output_paths["brand_lexicon"],
    index=False,
)
discharge_preannotations.to_csv(
    output_paths["discharge_annotations"],
    index=False,
)
discharge_review_queue.to_csv(
    output_paths["discharge_review"],
    index=False,
)
ingress_preannotations.to_csv(
    output_paths["ingress_annotations"],
    index=False,
)
ingress_review_queue.to_csv(
    output_paths["ingress_review"],
    index=False,
)

with output_paths["jsonl"].open("w", encoding="utf-8") as handle:
    for record in jsonl_records:
        handle.write(
            json.dumps(record, ensure_ascii=False)
            + "\n"
        )

checks = {
    "dataset_sha256": hashlib.sha256(
        DATASET_PATH.read_bytes()
    ).hexdigest(),
    "cohort_manifest_sha256": hashlib.sha256(
        COHORT_MANIFEST_PATH.read_bytes()
    ).hexdigest(),
    "selected_encounters": len(selected_ids),
    "unique_selected_encounters": len(set(selected_ids)),
    "source_reports": len(report_manifest),
    "automatic_discharge_annotations": len(
        discharge_preannotations
    ),
    "discharge_review_items": len(
        discharge_review_queue
    ),
    "automatic_ingress_annotations": len(
        ingress_preannotations
    ),
    "resolved_ingress_mappings": int(
        ingress_preannotations["mapping_status"]
        .isin(
            [
                "resolved_from_discharge_header",
                "direct_active_ingredient",
            ]
        )
        .sum()
    ),
    "ingress_review_items": len(
        ingress_review_queue
    ),
    "unique_discharge_ids": bool(
        discharge_preannotations["annotation_id"].is_unique
    ),
    "unique_ingress_ids": bool(
        ingress_preannotations["annotation_id"].is_unique
    ),
    "discharge_offsets_match_source": discharge_offsets_valid,
    "ingress_offsets_match_source": ingress_offsets_valid,
}

output_paths["checks"].write_text(
    json.dumps(checks, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

pd.DataFrame(
    {
        "file": output_paths.keys(),
        "path": [str(path.relative_to(PROJECT_ROOT)) for path in output_paths.values()],
    }
)

,file,path
0,report_manifest,data\annotations\generated\auto_report_manifest.csv
1,brand_lexicon,data\annotations\generated\auto_brand_ingredient_lexicon.csv
2,discharge_annotations,data\annotations\generated\auto_discharge_medications.csv
3,discharge_review,data\annotations\generated\auto_discharge_review_queue.csv
4,ingress_annotations,data\annotations\generated\auto_ingress_medications.csv
5,ingress_review,data\annotations\generated\auto_ingress_review_queue.csv
6,jsonl,data\annotations\generated\auto_annotations.jsonl
7,checks,data\annotations\generated\auto_checks.json


## Final summary

The review queues identify entries that require manual decisions. The automatic files should not be treated as final gold until those cases have been checked and incorporated.

In [14]:
summary = pd.Series(checks, name="value")
summary

dataset_sha256                     300318ecac0afb62c0e9424f6463bc75cd611246e3375aaabf0a651108707364
cohort_manifest_sha256             100d22fe4e3b343f689c766edeeebb1a252db89e337df2aa70317a2be9edabeb
selected_encounters                                                                              50
unique_selected_encounters                                                                       50
source_reports                                                                                  150
automatic_discharge_annotations                                                                 371
discharge_review_items                                                                            6
automatic_ingress_annotations                                                                   291
resolved_ingress_mappings                                                                       280
ingress_review_items                                                                             21


In [15]:
if not ingress_review_queue.empty:
    display(
        ingress_review_queue[
            ["reason", "source_text"]
        ]
        .value_counts("reason")
        .rename("count")
        .to_frame()
    )

if not discharge_review_queue.empty:
    display(
        discharge_review_queue[
            ["reason", "source_text"]
        ]
        .value_counts("reason")
        .rename("count")
        .to_frame()
    )

,count
reason,
explicit_no_therapy,6
unresolved,6
ambiguous_product_name,5
missing_colon_or_non_standard_list,2
non_drug_therapy,2


,count
reason,
unparsed_quoted_entry,4
no_regular_quoted_entries,2
